In [28]:
%pip install langchain-core langchain-groq pydantic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [29]:
import langchain_core
import langchain_groq
from pydantic import BaseModel, Field
from dotenv import load_dotenv

print("All libraries installed successfully!")

All libraries installed successfully!


In [30]:
from pathlib import Path

print(Path.cwd())

/home/radha/Desktop/scfp125033_mca_lab/lab4_Design_with_LLM


In [31]:
from pathlib import Path

for path in Path.home().glob("Desktop/scfp125033_mca_lab/**/.env"):
    print(path)

/home/radha/Desktop/scfp125033_mca_lab/myenv/.env
/home/radha/Desktop/scfp125033_mca_lab/lab3_prompt_engineering/.env
/home/radha/Desktop/scfp125033_mca_lab/.vscode/.env


In [32]:
from dotenv import load_dotenv

result = load_dotenv(
    "/home/radha/Desktop/scfp125033_mca_lab/myenv/.env"
)

print("Environment file loaded:", result)

Environment file loaded: True


In [33]:
import os

In [34]:
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

In [35]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0
)

print("Groq LLM configured successfully!")

Groq LLM configured successfully!


In [36]:
class ComplaintData(BaseModel):
    complaint: str = Field(description="The main complaint expressed by the customer")
    product_or_feature: str = Field(description="The product or feature mentioned in the review")
    sentiment: str = Field(description="Customer sentiment: Positive, Negative, or Neutral")

parser = PydanticOutputParser(pydantic_object=ComplaintData)

print("Structured complaint data model created successfully!")

Structured complaint data model created successfully!


In [37]:
extract_prompt = PromptTemplate(
    template="""Extract the following information from the customer review:
- complaint: The main problem reported by the customer.
- product_or_feature: The product or feature involved.
- sentiment: Positive, Negative, or Neutral.

Do not invent information.

Customer Review: {review_text}
{format_instructions}""",
    input_variables=["review_text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

In [38]:
extract_prompt

PromptTemplate(input_variables=['review_text'], input_types={}, partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"complaint": {"description": "The main complaint expressed by the customer", "title": "Complaint", "type": "string"}, "product_or_feature": {"description": "The product or feature mentioned in the review", "title": "Product Or Feature", "type": "string"}, "sentiment": {"description": "Customer sentiment: Positive, Negative, or Neutral", "title": "Sentiment", "type": "string"}}, "required": ["complaint", "product_or_

In [39]:
review_text = """
I purchased wireless headphones last week. The sound quality is good,
but the battery drains within two hours. I am disappointed with the battery performance.
"""

In [40]:
extract_chain = extract_prompt | llm | parser

In [41]:
structured_data = extract_chain.invoke({
    "review_text": review_text
})

In [42]:
print("CUSTOMER REVIEW:")
print(review_text)

print("\nEXTRACTED STRUCTURED DATA:")
print(structured_data.model_dump_json(indent=2))

CUSTOMER REVIEW:

I purchased wireless headphones last week. The sound quality is good,
but the battery drains within two hours. I am disappointed with the battery performance.


EXTRACTED STRUCTURED DATA:
{
  "complaint": "Battery drains within two hours and overall battery performance is disappointing",
  "product_or_feature": "Wireless headphones",
  "sentiment": "Negative"
}


In [43]:
ticket_prompt = ChatPromptTemplate.from_messages([
    ("system","""You are a customer support assistant.
        Create a concise, professional support ticket summary.
        Use only the provided structured data.
        Do not invent information."""
    ),
    ("human","""Create a support ticket using the following information:

        Complaint: {complaint}
        Product or Feature: {product_or_feature}
        Sentiment: {sentiment}

        Include the problem, affected product, and customer sentiment."""
    )
])

In [44]:
ticket_prompt

ChatPromptTemplate(input_variables=['complaint', 'product_or_feature', 'sentiment'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a customer support assistant.\n        Create a concise, professional support ticket summary.\n        Use only the provided structured data.\n        Do not invent information.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['complaint', 'product_or_feature', 'sentiment'], input_types={}, partial_variables={}, template='Create a support ticket using the following information:\n\n        Complaint: {complaint}\n        Product or Feature: {product_or_feature}\n        Sentiment: {sentiment}\n\n        Include the problem, affected product, and customer sentiment.'), additional_kwargs={})])

In [45]:
ticket_chain = ticket_prompt | llm | StrOutputParser()

In [46]:
ticket_summary = ticket_chain.invoke(
    structured_data.model_dump()
)

In [47]:
print("GENERATED SUPPORT TICKET")
print("========================")
print(ticket_summary)

GENERATED SUPPORT TICKET
**Support Ticket Summary**

- **Problem:** Battery drains within two hours and overall battery performance is disappointing.  
- **Affected Product:** Wireless headphones.  
- **Customer Sentiment:** Negative.


In [48]:
print("========== CUSTOMER REVIEW ==========")
print(review_text)

print("\n========== EXTRACTED DATA ==========")
print(structured_data.model_dump_json(indent=2))

print("\n========== SUPPORT TICKET ==========")
print(ticket_summary)

========== CUSTOMER REVIEW ==========

I purchased wireless headphones last week. The sound quality is good,
but the battery drains within two hours. I am disappointed with the battery performance.


========== EXTRACTED DATA ==========
{
  "complaint": "Battery drains within two hours and overall battery performance is disappointing",
  "product_or_feature": "Wireless headphones",
  "sentiment": "Negative"
}

========== SUPPORT TICKET ==========
**Support Ticket Summary**

- **Problem:** Battery drains within two hours and overall battery performance is disappointing.  
- **Affected Product:** Wireless headphones.  
- **Customer Sentiment:** Negative.
